Objetivo del estudio:
Analizar la información de libros, autores, editoriales y opiniones de usuarios para identificar patrones de lectura, preferencias de los clientes y características de los libros mejor valorados, con el fin de apoyar la creación de una propuesta de valor para un nuevo producto del mercado editorial digital.
análisis de datos
usuarios / calificaciones
apoyo a decisiones de negocio

In [4]:
import pandas as pd
from sqlalchemy import create_engine


db_config = {

 'user': 'practicum_student', # username
 'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7', # password
 'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
 'port': 5432, # connection port
 'db': 'data-analyst-final-project-db' # the name of the database

 }
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],

db_config['pwd'],

db_config['host'],
db_config['port'],

db_config['db'])

In [5]:
engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [6]:

query="""SELECT *
FROM books
LIMIT 5;
"""
pd.io.sql.read_sql(query, con = engine)


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


Relación:
Un libro tiene un autor
Un libro pertenece a una editorial
Tiene relación con la tabla authors		

In [7]:
query="""SELECT *
FROM authors
LIMIT 5;
"""
pd.io.sql.read_sql(query, con = engine)


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


Solo tiene relación con la tabla Books

In [ ]:
query="""SELECT *
FROM publishers
LIMIT 5;
"""
pd.io.sql.read_sql(query, con = engine)

Su relacion es con la tabla Books

In [ ]:
query="""SELECT *
FROM ratings
LIMIT 5;
"""
pd.io.sql.read_sql(query, con = engine)

Se relaciona con la tabla Books y con la tabla Reviews

In [ ]:
query="""SELECT *
FROM reviews
LIMIT 5;
"""
pd.io.sql.read_sql(query, con = engine)

Su relación es con la tabla ratings

### Encuentra el número de libros publicados después del 1 de enero de 2000.

In [12]:
query = """
SELECT COUNT(*) AS total_books
FROM books
WHERE publication_date >= '2000-01-01';
"""
pd.io.sql.read_sql(query, con = engine)

,total_books
0,821


Se publicaron 821 libros despues del 1 de Enero del 2020.

COUNT(*) → cuenta cuántos libros hay
🔹 publication_date > '2000-01-01' → filtra solo los libros publicados después de esa fecha
🔹 AS total_books → nombre claro para la columna. 

### Encuentra el número de reseñas de usuarios y la calificación promedio para cada libro.

In [9]:
query = """
SELECT
    b.book_id,
    b.title,
    COUNT(DISTINCT r.review_id) AS reviews_count,
    AVG(rt.rating) AS avg_rating
FROM books b
LEFT JOIN reviews r ON b.book_id = r.book_id
LEFT JOIN ratings rt ON b.book_id = rt.book_id
GROUP BY b.book_id, b.title
ORDER BY reviews_count DESC;
"""
pd.read_sql(query, con=engine)

,book_id,title,reviews_count,avg_rating
0,948,Twilight (Twilight #1),7,3.662500
1,963,Water for Elephants,6,3.977273
2,734,The Glass Castle,6,4.206897
3,302,Harry Potter and the Prisoner of Azkaban (Harr...,6,4.414634
4,695,The Curious Incident of the Dog in the Night-Time,6,4.081081
...,...,...,...,...
995,83,Anne Rice's The Vampire Lestat: A Graphic Novel,0,3.666667
996,808,The Natural Way to Draw,0,3.000000
997,672,The Cat in the Hat and Other Dr. Seuss Favorites,0,5.000000
998,221,Essential Tales and Poems,0,4.000000


¿Qué estamos sacando?
Para cada libro:
cuántas reseñas escritas tiene
cuál es su calificación promedio

🔹 Claves del SQL
LEFT JOIN → incluye libros aunque no tengan reseñas o calificaciones
COUNT(DISTINCT r.review_id) → cuenta reseñas (sin duplicar)
AVG(rt.rating) → promedio de calificaciones
GROUP BY → un resultado por libro
ORDER BY reviews_count DESC → libros más comentados primero

Se observa que algunos libros concentran un mayor número de reseñas, lo que indica mayor participación de los usuarios. Además, la calificación promedio permite identificar libros mejor valorados, información clave para destacar contenidos populares y de alta calidad en la propuesta de valor del producto.

### Identifica la editorial que ha publicado el mayor número de libros con más de 50 páginas (esto te ayudará a excluir folletos y publicaciones similares de tu análisis).

In [10]:
query = """
SELECT
    p.publisher,
    COUNT(b.book_id) AS books_count
FROM books b
JOIN publishers p ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY books_count DESC
LIMIT 1;
"""
pd.read_sql(query, con=engine)

,publisher,books_count
0,Penguin Books,42


num_pages > 50 → excluye folletos y publicaciones cortas
JOIN publishers → obtiene el nombre de la editorial
COUNT(b.book_id) → cuenta libros reales
ORDER BY ... DESC → de mayor a menor
LIMIT 1 → solo la editorial líder

La editorial identificada es la que concentra el mayor número de libros con más de 50 páginas, lo que indica una producción predominante de libros completos y no publicaciones breves. Esta información es relevante para priorizar editoriales con catálogos más extensos y consistentes dentro del nuevo producto.

### Identifica al autor que tiene la más alta calificación promedio del libro: mira solo los libros con al menos 50 calificaciones.

In [11]:
query = """
SELECT
    a.author,
    AVG(rt.rating) AS avg_rating,
    COUNT(rt.rating) AS ratings_count
FROM books b
JOIN authors a ON b.author_id = a.author_id
JOIN ratings rt ON b.book_id = rt.book_id
GROUP BY a.author
HAVING COUNT(rt.rating) >= 50
ORDER BY avg_rating DESC
LIMIT 1;
"""
pd.read_sql(query, con=engine)

,author,avg_rating,ratings_count
0,Diana Gabaldon,4.3,50


In [ ]:
JOIN

books → conecta libros
authors → obtiene el nombre del autor
ratings → obtiene las calificaciones
AVG(rt.rating)
Calcula la calificación promedio de los libros del autor.
 COUNT(rt.rating) >= 50
Condición clave del ejercicio
Solo se consideran autores cuyos libros tengan al menos 50 calificaciones
Esto evita resultados poco confiables.
Esta condición NO va en WHERE
Va en HAVING porque usa una agregación (COUNT)
ORDER BY avg_rating DESC
Ordena de mayor a menor calificación promedio.
LIMIT 1
Devuelve solo el autor con la mejor calificación promedio.

El autor identificado presenta la calificación promedio más alta considerando únicamente libros con al menos 50 calificaciones, lo que indica una valoración consistentemente positiva por parte de los usuarios y un alto nivel de confiabilidad en las opiniones.

### Encuentra el número promedio de reseñas de texto entre los usuarios que calificaron más de 50 libros.

In [14]:
query = """
WITH active_users AS (
    SELECT
        username
    FROM ratings
    GROUP BY username
    HAVING COUNT(book_id) > 50
),
reviews_per_user AS (
    SELECT
        r.username,
        COUNT(r.review_id) AS reviews_count
    FROM reviews r
    JOIN active_users au ON r.username = au.username
    GROUP BY r.username
)
SELECT
    AVG(reviews_count) AS avg_reviews_per_user
FROM reviews_per_user;
"""

pd.read_sql(query, con=engine)

,avg_reviews_per_user
0,24.333333


active_users
Identifica usuarios que calificaron más de 50 libros
Usa HAVING porque es una condición con COUNT
🔹 reviews_per_user
Cuenta cuántas reseñas escritas tiene cada uno de esos usuarios
🔹 AVG(reviews_count)
Calcula el promedio final

En promedio, los usuarios que califican más de 50 libros escriben un número moderado de reseñas de texto, lo que sugiere que, aunque son altamente activos calificando libros, solo una parte de sus interacciones incluye comentarios detallados.